# WP3 Africa Hazard Ranking — Dataset Notebook 01  
## WorldRiskIndex (WRI) — country-level indicator extraction (up to 2025)

**Purpose of this notebook (dataset-specific):**  
Extract a *country-level* set of **WorldRiskIndex** indicators for the WP3 Africa hazard ranking workflow, harmonised to the shared **long-format contract** and ready to be appended (via *upsert*) into the shared intermediate CSV.

**Key project decisions reflected here:**
- **Scope countries:** *DataBank countries only*.  
- **Hazards:** fixed at 9 (plus a special **All Hazards** category for cross-hazard indicators).
- **Presence:** not a scoring dimension; it is a **coverage flag** only.
- **This dataset:** use the annual series **through 2025**; for the shared intermediate scoring table we extract the **latest available year (2025)** as a cross-sectional snapshot.

> If you later decide to use a different WRI year or a multi-year summary, change `WRI_YEAR_FOR_SCORING` in the config cell below.

---

## Inputs expected
1) `ISO3_Regions.csv`: columns **Region, Country, ISO3.Code**.  
2) `worldriskindex-datasets.zip` containing yearly CSVs (2000–2025).

---

## Outputs written
- `data/intermediate/worldriskindex_extracted_2025_long.csv`  (this notebook’s extracted indicators, long format)
- `data/intermediate/worldriskindex_coverage_2025.csv`        (presence/coverage flags by hazard)
- `data/intermediate/wp3_country_indicators_long.csv`         (shared master; this notebook upserts into it)

---

## Indicator selection
We only extract the following WRI fields for WP3:

### Hazard-specific (Scale proxies)
- **Drought / Scale** → `EI_06`
- **Riverine Flooding Continued & Cluster / Scale** → `EI_04`
- **Storm Surge / Scale** → `EI_03`  *(WRI “coastal flooding” exposure proxy)*
- **Wind / Scale** → `EI_05`          *(WRI “cyclone” exposure proxy)*

### Cross-hazard
- **All Hazards / Scale** → `E`  *(Exposure sphere; overall exposure to WRI hazards)*
- **All Hazards / Future relevance** → `A` *(Lack of adaptive capacities; used here as a future-relevance proxy as per agreement)*

### Presence / coverage flags
Presence for each hazard above is derived as **value-not-missing** for the corresponding column (e.g., `EI_06` present → drought evidence present in WRI).

In [16]:
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile

# -----------------------------
# Config — edit paths if needed
# -----------------------------
PROJECT_ROOT = Path(".").resolve()  # set to repo root when running locally

# Inputs (expected repo locations)
COUNTRY_SCOPE_PATH = PROJECT_ROOT.parent / "data" / "raw" / "scope" / "ISO3_Regions.csv"
WRI_ZIP_PATH       = PROJECT_ROOT.parent / "data" / "raw" / "worldriskindex" / "worldriskindex-datasets.zip"

# Fallbacks for quick local testing (won't exist in most repos; safe to ignore)
FALLBACK_SCOPE = Path("/mnt/data/ISO3_Regions.csv")
FALLBACK_WRI   = Path("/mnt/data/worldriskindex-datasets.zip")

if not COUNTRY_SCOPE_PATH.exists() and FALLBACK_SCOPE.exists():
    COUNTRY_SCOPE_PATH = FALLBACK_SCOPE
if not WRI_ZIP_PATH.exists() and FALLBACK_WRI.exists():
    WRI_ZIP_PATH = FALLBACK_WRI

# Scoring-year choice for WRI (latest available = 2025)
WRI_YEAR_FOR_SCORING = 2025

# Outputs
OUT_DIR = PROJECT_ROOT.parent / "data" / "intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WRI_EXTRACT_LONG_PATH   = OUT_DIR / f"worldriskindex_extracted_{WRI_YEAR_FOR_SCORING}_long.csv"
WRI_COVERAGE_PATH       = OUT_DIR / f"worldriskindex_coverage_{WRI_YEAR_FOR_SCORING}.csv"
MASTER_LONG_PATH        = OUT_DIR / "wp3_country_indicators_long.csv"

print("PROJECT_ROOT:", PROJECT_ROOT.parent)
print("COUNTRY_SCOPE_PATH:", COUNTRY_SCOPE_PATH, "exists:", COUNTRY_SCOPE_PATH.exists())
print("WRI_ZIP_PATH:", WRI_ZIP_PATH, "exists:", WRI_ZIP_PATH.exists())
print("Outputs to:", OUT_DIR)

PROJECT_ROOT: C:\pipelines\sewa-multihazar
COUNTRY_SCOPE_PATH: C:\pipelines\sewa-multihazar\data\raw\scope\ISO3_Regions.csv exists: True
WRI_ZIP_PATH: C:\pipelines\sewa-multihazar\data\raw\worldriskindex\worldriskindex-datasets.zip exists: True
Outputs to: C:\pipelines\sewa-multihazar\data\intermediate


## Global setup — the long-format contract (shared across all dataset notebooks)

All dataset notebooks **must** output indicators in this shared schema, then **upsert** into the same master file.

### Master long-format schema
Required columns:

- `iso3` (ISO3 code, uppercase)
- `country_name` (display name from scope list)
- `region` (your project region)
- `hazard` (one of the 9 hazards, or **All Hazards**)
- `dimension` (one of: `Prevalence`, `Scale`, `Impact`, `Cascade impacts`, `Future relevance`)
- `source` (dataset name, here: `WorldRiskIndex`)
- `indicator_id` (stable machine ID)
- `indicator_name` (human-readable)
- `value_raw` (numeric)
- `unit_raw` (string; for WRI we store `index_0_100`)
- `year` (numeric; for WRI this is `2025`)
- `time_window` (string; for WRI this is `annual_snapshot`)
- `notes` (string; mapping & assumptions)

### Upsert key (must be unique)
`iso3 + hazard + dimension + source + indicator_id + year`

### Coverage (presence flags)
Presence is a **separate output** (coverage matrix) and **must not** be mixed into the 5 scoring dimensions.

In [17]:
# Canonical enums (keep consistent across notebooks)
HAZARDS_9 = [
    "Drought",
    "Flash Flooding",
    "Riverine Flooding Continued & Cluster",
    "Heatwave",
    "Storm Surge",
    "Wind",
    "Thunderstorm",
    "Wildfires",
    "Dust",
]
DIMENSIONS_5 = ["Prevalence", "Scale", "Impact", "Cascade impacts", "Future relevance"]

SOURCE_NAME = "WorldRiskIndex"

LONG_COLUMNS = [
    "iso3","country_name","region","hazard","dimension","source",
    "indicator_id","indicator_name","value_raw","unit_raw",
    "year","time_window","notes"
]

UPSERT_KEY = ["iso3","hazard","dimension","source","indicator_id","year"]

def assert_contract(df: pd.DataFrame) -> None:
    missing_cols = [c for c in LONG_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Contract violation: missing columns {missing_cols}")

    # Basic enum validation
    bad_dims = sorted(set(df["dimension"]) - set(DIMENSIONS_5))
    if bad_dims:
        raise ValueError(f"Contract violation: unknown dimensions {bad_dims}")

    # Hazards allow All Hazards
    allowed_haz = set(HAZARDS_9) | {"All Hazards"}
    bad_haz = sorted(set(df["hazard"]) - allowed_haz)
    if bad_haz:
        raise ValueError(f"Contract violation: unknown hazards {bad_haz}")

    # Upsert-key uniqueness
    dup = df.duplicated(UPSERT_KEY, keep=False)
    if dup.any():
        example = df.loc[dup, UPSERT_KEY].head(10)
        raise ValueError(f"Contract violation: duplicate UPSERT keys. Examples:\n{example}")

def upsert_to_master(new_df: pd.DataFrame, master_path: Path) -> pd.DataFrame:
    """Upsert new_df into master CSV using UPSERT_KEY."""
    assert_contract(new_df)

    if master_path.exists():
        master = pd.read_csv(master_path)
        # Ensure all contract columns exist in master (backfill if needed)
        for col in LONG_COLUMNS:
            if col not in master.columns:
                master[col] = np.nan
        master = master[LONG_COLUMNS]

        # Drop any existing rows that match new keys
        master_key = master[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        new_key    = new_df[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        master = master.loc[~master_key.isin(set(new_key))].copy()

        out = pd.concat([master, new_df[LONG_COLUMNS]], ignore_index=True)
    else:
        out = new_df[LONG_COLUMNS].copy()

    out.to_csv(master_path, index=False)
    return out

## Step 1 — Load country scope (ISO3 + region) and clean it

In [18]:
scope_raw = pd.read_csv(COUNTRY_SCOPE_PATH)

# Normalise headers with non-breaking spaces etc.
scope_raw.columns = [c.replace("\xa0"," ").strip() for c in scope_raw.columns]

# Expect: Region, Country, ISO3.Code
scope = scope_raw.copy()
scope["Region"] = scope["Region"].replace({np.nan: None})
scope["Region"] = scope["Region"].ffill()

scope["Region"] = scope["Region"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["Country"] = scope["Country"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["ISO3.Code"] = scope["ISO3.Code"].astype(str).str.strip().str.upper()

# Clean region names (keep your four groups, but make them readable)
scope["Region"] = scope["Region"].str.replace(r"\s+Country$", "", regex=True).str.strip()
scope["Region"] = scope["Region"].replace({
    "Southern Africaincl. Indian Ocean Islands Country": "Southern Africa (incl. Indian Ocean Islands)",
    "Southern Africaincl. Indian Ocean Islands": "Southern Africa (incl. Indian Ocean Islands)",
})

scope = scope.rename(columns={"Region":"region","Country":"country_name","ISO3.Code":"iso3"})
scope = scope[["iso3","country_name","region"]].drop_duplicates()

# Minimal sanity checks
if scope["iso3"].isna().any():
    raise ValueError("Scope list contains missing ISO3 codes.")
if scope["iso3"].duplicated().any():
    dups = scope.loc[scope["iso3"].duplicated(), "iso3"].tolist()
    raise ValueError(f"Duplicate ISO3 in scope list: {dups}")

print("Scope countries:", len(scope))
print("Regions:", sorted(scope["region"].unique().tolist()))
scope.head(10)

Scope countries: 47
Regions: ['Central Africa', 'East Africa', 'Southern Africa (incl. Indian Ocean Islands)', 'West Africa']


,iso3,country_name,region
0,NGA,Nigeria,West Africa
1,GHA,Ghana,West Africa
2,SEN,Senegal,West Africa
3,BFA,Burkina Faso,West Africa
4,MLI,Mali,West Africa
5,NER,Niger,West Africa
6,CIV,Côte d'Ivoire,West Africa
7,BEN,Benin,West Africa
8,TGO,Togo,West Africa
9,SLE,Sierra Leone,West Africa


## Step 2 — Load WorldRiskIndex data

In [19]:
def load_wri_year_from_zip(zip_path: Path, year: int) -> pd.DataFrame:
    target = f"worldriskindex-datasets/worldriskindex-{year}.csv"
    with zipfile.ZipFile(zip_path) as zf:
        if target not in zf.namelist():
            raise FileNotFoundError(f"{target} not found inside {zip_path.name}")
        with zf.open(target) as f:
            df = pd.read_csv(f)
    return df

wri = load_wri_year_from_zip(WRI_ZIP_PATH, WRI_YEAR_FOR_SCORING)

# Normalise key columns
wri = wri.rename(columns={"WRI.Country":"country_wri", "ISO3.Code":"iso3"})
wri["iso3"] = wri["iso3"].astype(str).str.strip().str.upper()

print("WRI rows:", len(wri), "cols:", wri.shape[1])
wri[["country_wri","iso3","Year"]].head()

WRI rows: 193 cols: 248


,country_wri,iso3,Year
0,Afghanistan,AFG,2025
1,Albania,ALB,2025
2,Algeria,DZA,2025
3,Andorra,AND,2025
4,Angola,AGO,2025


## Step 3 — Join diagnostics & handling of missing territories

In [20]:
wri_iso = set(wri["iso3"].unique())
scope_iso = set(scope["iso3"].unique())

missing_in_wri = sorted(list(scope_iso - wri_iso))
extra_in_wri = sorted(list(wri_iso - scope_iso))

print("Missing ISO3 in WRI (from scope):", missing_in_wri)
print("Extra ISO3 in WRI (not in scope):", len(extra_in_wri))

# Join (left join keeps all scope countries, even if WRI missing)
joined = scope.merge(wri, on="iso3", how="left", validate="1:1")

joined_missing = joined.loc[joined["country_wri"].isna(), ["iso3","country_name","region"]]
joined_missing

Missing ISO3 in WRI (from scope): ['REU']
Extra ISO3 in WRI (not in scope): 147


,iso3,country_name,region
46,REU,Réunion (France),Southern Africa (incl. Indian Ocean Islands)


## Step 4 — Extract agreed indicators

**Mapping implemented (explicit assumptions):**
- WRI `EI_06` (drought exposure) is used as **Drought / Scale**.
- WRI `EI_04` (riverine flooding exposure) is used as **Riverine Flooding Continued & Cluster / Scale**.
- WRI `EI_03` (coastal flooding exposure) is used as **Storm Surge / Scale** *(proxy; WRI uses “coastal flooding”)*.
- WRI `EI_05` (cyclone exposure) is used as **Wind / Scale** *(proxy; cyclone exposure is used as wind-storm indicator)*.
- WRI `E` (overall exposure sphere) is used as **All Hazards / Scale**.
- WRI `A` (lack of adaptive capacities) is used as **All Hazards / Future relevance** (as agreed in this project).

**Unit convention for WRI indicators:**  
WRI values are provided as an **index on a 0–100 scale** (store as `index_0_100`); normalisation to 0–1 happens later in scoring notebooks.

In [21]:
INDICATORS = [
    dict(hazard="Drought", dimension="Scale", wri_col="EI_06",
         indicator_id="WRI.EI_06", indicator_name="Exposure to droughts (EI_06)"),
    dict(hazard="Riverine Flooding Continued & Cluster", dimension="Scale", wri_col="EI_04",
         indicator_id="WRI.EI_04", indicator_name="Exposure to riverine flooding (EI_04)"),
    dict(hazard="Storm Surge", dimension="Scale", wri_col="EI_03",
         indicator_id="WRI.EI_03", indicator_name="Exposure to coastal flooding (EI_03) — proxy for storm surge"),
    dict(hazard="Wind", dimension="Scale", wri_col="EI_05",
         indicator_id="WRI.EI_05", indicator_name="Exposure to cyclones (EI_05) — proxy for wind"),
    dict(hazard="All Hazards", dimension="Scale", wri_col="E",
         indicator_id="WRI.E", indicator_name="Exposure sphere (E) — overall hazard exposure"),
    dict(hazard="All Hazards", dimension="Future relevance", wri_col="A",
         indicator_id="WRI.A", indicator_name="Lack of adaptive capacities (A) — future relevance proxy"),
]

missing_cols = [spec["wri_col"] for spec in INDICATORS if spec["wri_col"] not in joined.columns]
if missing_cols:
    raise ValueError(f"Missing required WRI columns in joined dataframe: {missing_cols}")

def make_long(df_joined: pd.DataFrame, indicators: list, year: int) -> pd.DataFrame:
    rows = []
    for spec in indicators:
        tmp = df_joined[["iso3","country_name","region", spec["wri_col"]]].copy()
        tmp = tmp.rename(columns={spec["wri_col"]:"value_raw"})
        tmp["hazard"] = spec["hazard"]
        tmp["dimension"] = spec["dimension"]
        tmp["source"] = SOURCE_NAME
        tmp["indicator_id"] = spec["indicator_id"]
        tmp["indicator_name"] = spec["indicator_name"]
        tmp["unit_raw"] = "index_0_100"
        tmp["year"] = year
        tmp["time_window"] = "annual_snapshot"
        tmp["notes"] = f"WRI column '{spec['wri_col']}' mapped to {spec['hazard']} / {spec['dimension']}."
        rows.append(tmp)

    out = pd.concat(rows, ignore_index=True)
    out = out.loc[~out["value_raw"].isna()].copy()  # drop missing values (coverage handled separately)
    return out

wri_long = make_long(joined, INDICATORS, WRI_YEAR_FOR_SCORING)

assert_contract(wri_long)

print("Extracted rows:", len(wri_long))
wri_long.head(12)

Extracted rows: 276


,iso3,country_name,region,value_raw,hazard,dimension,source,indicator_id,indicator_name,unit_raw,year,time_window,notes
0,NGA,Nigeria,West Africa,3.21,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
1,GHA,Ghana,West Africa,3.08,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
2,SEN,Senegal,West Africa,2.00,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
3,BFA,Burkina Faso,West Africa,2.80,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
4,MLI,Mali,West Africa,2.84,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
5,NER,Niger,West Africa,2.33,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
6,CIV,Côte d'Ivoire,West Africa,3.36,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
7,BEN,Benin,West Africa,2.82,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
8,TGO,Togo,West Africa,2.12,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.
9,SLE,Sierra Leone,West Africa,3.66,Drought,Scale,WorldRiskIndex,WRI.EI_06,Exposure to droughts (EI_06),index_0_100,2025,annual_snapshot,WRI column 'EI_06' mapped to Drought / Scale.


## Step 5 — Create coverage (presence) flags

In [22]:
coverage_specs = [
    ("Drought", "EI_06"),
    ("Riverine Flooding Continued & Cluster", "EI_04"),
    ("Storm Surge", "EI_03"),
    ("Wind", "EI_05"),
    ("All Hazards", "E"),
    ("All Hazards", "A"),
]

cov_rows = []
for haz, col in coverage_specs:
    present = joined[["iso3","country_name","region", col]].copy()
    present["hazard"] = haz
    present["wri_col"] = col
    present["present"] = ~present[col].isna()
    cov_rows.append(present[["iso3","country_name","region","hazard","wri_col","present"]])

coverage = pd.concat(cov_rows, ignore_index=True)
coverage.to_csv(WRI_COVERAGE_PATH, index=False)

coverage.groupby(["hazard","present"]).size().unstack(fill_value=0)

present,False,True
hazard,,
All Hazards,2,92
Drought,1,46
Riverine Flooding Continued & Cluster,1,46
Storm Surge,1,46
Wind,1,46


## Step 6 — Write outputs and upsert into the shared master CSV

In [23]:
wri_long.to_csv(WRI_EXTRACT_LONG_PATH, index=False)

master = upsert_to_master(wri_long, MASTER_LONG_PATH)

print("Wrote:", WRI_EXTRACT_LONG_PATH)
print("Wrote:", WRI_COVERAGE_PATH)
print("Updated master:", MASTER_LONG_PATH, "rows:", len(master))

master.tail(10)

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\worldriskindex_extracted_2025_long.csv
Wrote: C:\pipelines\sewa-multihazar\data\intermediate\worldriskindex_coverage_2025.csv
Updated master: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv rows: 276


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
271,NAM,Namibia,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,56.57,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
272,MOZ,Mozambique,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,66.79,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
273,AGO,Angola,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,68.91,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
274,MWI,Malawi,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,67.30,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
275,LSO,Lesotho,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,63.21,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
276,SWZ,Eswatini (Swaziland),Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,55.93,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
277,MDG,Madagascar,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,71.62,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
278,MUS,Mauritius,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,39.28,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
279,SYC,Seychelles,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,34.64,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...
280,COM,Comoros,Southern Africa (incl. Indian Ocean Islands),All Hazards,Future relevance,WorldRiskIndex,WRI.A,Lack of adaptive capacities (A) — future relev...,59.08,index_0_100,2025,annual_snapshot,WRI column 'A' mapped to All Hazards / Future ...


## Flags

1) **Réunion (REU) is not present in WorldRiskIndex (country dataset targets UN member states).**  
   This notebook keeps Réunion in scope  but will show it as missing for WRI indicators and coverage.

2) **Proxy mappings require careful wording downstream:**
   - WRI “coastal flooding” exposure is used as a proxy for **Storm Surge**.
   - WRI “cyclone” exposure is used as a proxy for **Wind**.

3) **Avoiding overweighting via column count:**  
   This notebook intentionally extracts only the agreed WRI fields, because later scoring uses **equal weight per indicator**.

If you later decide to use WRI’s `S` (susceptibility) or `C` (lack of coping capacities) as **Cascade impacts** proxies, do it as an explicit, documented choice in a future revision.